<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w05_model_training_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

## Search Impressions Decline — momentum features + grouped tuning

This notebook rebuilds the Week 5 model from the data query onward.

The main changes are:

1. Keep the March 1–15 feature window and March 16–31 outcome window.
2. Split March 1–15 into **days 1–7** and **days 8–15** so the model can see direction, not only level.
3. Keep a **client holdout test set** for the final evaluation only.
4. Tune hyperparameters with **GroupKFold inside the training clients only**.
5. Compare the final models against the Week 4 rule baseline on the same held-out clients and the same ranking metrics.

The target remains:

`is_declining_proxy = 1` when average daily impressions in March 16–31 are more than 20% lower than average daily impressions in March 1–15.

## 1. Method choice and why

I will compare Logistic Regression, Decision Tree, and Random Forest.

This is a binary classification problem, but the decision I care about is ranking: which pages should be reviewed first for decline risk. For that reason, the models will be evaluated mainly with **Precision@100**, with Precision@10, Average Precision, and ROC-AUC as supporting metrics.

Logistic Regression is the readable reference. Decision Tree captures simple non-linear rules. Random Forest can capture more complex interactions between level and momentum signals.

The final test set is not used to choose hyperparameters.

In [ ]:
%pip install -q duckdb huggingface_hub pandas numpy scikit-learn matplotlib

In [ ]:
from google.colab import userdata

import duckdb
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

pandas: 2.2.3
numpy: 2.1.3
scikit-learn: 1.6.1


In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Connected successfully.")

Connected successfully.


## 2. Build the modeling table

The feature window is still March 1–15, but it is divided into two sub-windows:

- **Early:** March 1–7
- **Late:** March 8–15

The future outcome remains March 16–31.

Using daily averages avoids a fake difference caused by comparing a 7-day window with an 8-day window.

In [ ]:
query = f'''
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    -- -------------------------
    -- Early feature window: Mar 1-7
    -- -------------------------
    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07'
             THEN COALESCE(gsc_impressions, 0) ELSE 0 END
    ) AS imp_early,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07'
             THEN COALESCE(gsc_clicks, 0) ELSE 0 END
    ) AS clicks_early,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07'
             THEN COALESCE(gsc_sum_position, 0) ELSE 0 END
    ) AS sum_position_early,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07'
                  AND COALESCE(gsc_data_available, FALSE)
             THEN 1 ELSE 0 END
    ) AS gsc_days_early,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07'
                  AND COALESCE(gsc_impressions, 0) > 0
             THEN 1 ELSE 0 END
    ) AS active_days_early,

    -- -------------------------
    -- Late feature window: Mar 8-15
    -- -------------------------
    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15'
             THEN COALESCE(gsc_impressions, 0) ELSE 0 END
    ) AS imp_late,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15'
             THEN COALESCE(gsc_clicks, 0) ELSE 0 END
    ) AS clicks_late,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15'
             THEN COALESCE(gsc_sum_position, 0) ELSE 0 END
    ) AS sum_position_late,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15'
                  AND COALESCE(gsc_data_available, FALSE)
             THEN 1 ELSE 0 END
    ) AS gsc_days_late,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15'
                  AND COALESCE(gsc_impressions, 0) > 0
             THEN 1 ELSE 0 END
    ) AS active_days_late,

    -- -------------------------
    -- Future outcome: Mar 16-31
    -- -------------------------
    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
             THEN COALESCE(gsc_impressions, 0) ELSE 0 END
    ) AS imp_future,

    SUM(
        CASE WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                  AND COALESCE(gsc_data_available, FALSE)
             THEN 1 ELSE 0 END
    ) AS gsc_days_future

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'

GROUP BY
    client_hash_id,
    content_hash_id
'''

df = con.execute(query).df()

print("Rows returned:", len(df))
print("Clients:", df["client_id"].nunique())

display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows returned: 331437
Clients: 55


,client_id,content_id,imp_early,clicks_early,sum_position_early,gsc_days_early,active_days_early,imp_late,clicks_late,sum_position_late,gsc_days_late,active_days_late,imp_future,gsc_days_future
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2437.0,4.0,14573.0,7.0,7.0,1736.0,2.0,11571.0,8.0,8.0,2350.0,16.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,119.0,0.0,577.0,7.0,7.0,126.0,0.0,424.0,8.0,8.0,208.0,16.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,1994.0,3.0,12785.0,7.0,7.0,1711.0,0.0,10548.0,8.0,8.0,1925.0,16.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,1415.0,1.0,10588.0,7.0,7.0,1025.0,7.0,7397.0,8.0,8.0,2504.0,16.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,2.0,0.0,17.0,1.0,1.0,12.0,0.0,127.0,8.0,8.0,28.0,12.0


In [ ]:
# Keep pages with complete GSC coverage across all three windows.
clean_df = df[
    (df["gsc_days_early"] == 7)
    & (df["gsc_days_late"] == 8)
    & (df["gsc_days_future"] == 16)
].copy()

# Whole first-half level features.
clean_df["imp_first_half"] = clean_df["imp_early"] + clean_df["imp_late"]
clean_df["clicks_first_half"] = clean_df["clicks_early"] + clean_df["clicks_late"]
clean_df["sum_position_first_half"] = (
    clean_df["sum_position_early"] + clean_df["sum_position_late"]
)
clean_df["active_days_first_half"] = (
    clean_df["active_days_early"] + clean_df["active_days_late"]
)

# The target needs a non-zero first-half denominator.
clean_df = clean_df[clean_df["imp_first_half"] > 0].copy()

# Daily averages: 15-day feature window vs 16-day future window.
clean_df["avg_daily_imp_first_half"] = clean_df["imp_first_half"] / 15.0
clean_df["avg_daily_imp_future"] = clean_df["imp_future"] / 16.0

clean_df["impression_change_pct"] = (
    (
        clean_df["avg_daily_imp_future"]
        - clean_df["avg_daily_imp_first_half"]
    )
    / clean_df["avg_daily_imp_first_half"]
) * 100.0

clean_df["is_declining_proxy"] = (
    clean_df["impression_change_pct"] < -20
).astype(int)

print("Eligible pages:", len(clean_df))
print("Clients:", clean_df["client_id"].nunique())
print("Decline rate:", f'{clean_df["is_declining_proxy"].mean():.1%}')

Eligible pages: 61796
Clients: 34
Decline rate: 32.4%


## Momentum features

The old feature set mostly described **level** over the full 15 days. These features add **direction** inside the same pre-outcome window.

For impressions, the percentage change uses daily averages and a fixed minimum denominator of 1 impression/day. This prevents tiny early values from creating extreme ratios.

For CTR, I use percentage-point change instead of percentage change because CTR can be zero.

For position, a positive change means the average ranking position became worse.

In [ ]:
# -------------------------
# Level features
# -------------------------
clean_df["ctr_first_half"] = (
    clean_df["clicks_first_half"] / clean_df["imp_first_half"]
) * 100.0

clean_df["avg_position_first_half"] = np.where(
    clean_df["imp_first_half"] > 0,
    clean_df["sum_position_first_half"] / clean_df["imp_first_half"],
    np.nan,
)

clean_df["active_rate_first_half"] = clean_df["active_days_first_half"] / 15.0

# Log transforms reduce the effect of very large traffic values for the linear model.
clean_df["log_imp_first_half"] = np.log1p(clean_df["imp_first_half"])
clean_df["log_clicks_first_half"] = np.log1p(clean_df["clicks_first_half"])

# -------------------------
# Early vs late daily levels
# -------------------------
clean_df["avg_daily_imp_early"] = clean_df["imp_early"] / 7.0
clean_df["avg_daily_imp_late"] = clean_df["imp_late"] / 8.0

clean_df["avg_daily_clicks_early"] = clean_df["clicks_early"] / 7.0
clean_df["avg_daily_clicks_late"] = clean_df["clicks_late"] / 8.0

clean_df["ctr_early"] = np.where(
    clean_df["imp_early"] > 0,
    (clean_df["clicks_early"] / clean_df["imp_early"]) * 100.0,
    0.0,
)

clean_df["ctr_late"] = np.where(
    clean_df["imp_late"] > 0,
    (clean_df["clicks_late"] / clean_df["imp_late"]) * 100.0,
    0.0,
)

clean_df["avg_position_early"] = np.where(
    clean_df["imp_early"] > 0,
    clean_df["sum_position_early"] / clean_df["imp_early"],
    np.nan,
)

clean_df["avg_position_late"] = np.where(
    clean_df["imp_late"] > 0,
    clean_df["sum_position_late"] / clean_df["imp_late"],
    np.nan,
)

# -------------------------
# Direction / momentum features
# -------------------------
early_imp_den = np.maximum(clean_df["avg_daily_imp_early"], 1.0)

clean_df["imp_momentum_pct"] = (
    (
        clean_df["avg_daily_imp_late"]
        - clean_df["avg_daily_imp_early"]
    )
    / early_imp_den
) * 100.0

# A second, smoother representation of impression momentum.
clean_df["imp_momentum_log"] = (
    np.log1p(clean_df["avg_daily_imp_late"])
    - np.log1p(clean_df["avg_daily_imp_early"])
)

clean_df["click_change_per_day"] = (
    clean_df["avg_daily_clicks_late"]
    - clean_df["avg_daily_clicks_early"]
)

clean_df["ctr_change_pp"] = clean_df["ctr_late"] - clean_df["ctr_early"]

clean_df["has_position_momentum"] = (
    clean_df["avg_position_early"].notna()
    & clean_df["avg_position_late"].notna()
).astype(int)

clean_df["position_change"] = (
    clean_df["avg_position_late"] - clean_df["avg_position_early"]
).fillna(0.0)

clean_df["active_rate_change"] = (
    clean_df["active_days_late"] / 8.0
    - clean_df["active_days_early"] / 7.0
)

# Keep the same basic position eligibility used in the previous model.
model_df = clean_df[
    clean_df["avg_position_first_half"].notna()
    & (clean_df["avg_position_first_half"] > 0)
].copy().reset_index(drop=True)

feature_cols = [
    # Level
    "log_imp_first_half",
    "log_clicks_first_half",
    "ctr_first_half",
    "avg_position_first_half",
    "active_rate_first_half",

    # Momentum
    "imp_momentum_pct",
    "imp_momentum_log",
    "click_change_per_day",
    "ctr_change_pp",
    "position_change",
    "active_rate_change",
    "has_position_momentum",
]

target_col = "is_declining_proxy"

print("Modeling pages:", len(model_df))
print("Clients:", model_df["client_id"].nunique())
print("Features:", len(feature_cols))
print("Missing feature values:", model_df[feature_cols].isna().sum().sum())

display(
    model_df[
        [
            "content_id",
            "imp_first_half",
            "avg_daily_imp_early",
            "avg_daily_imp_late",
            "imp_momentum_pct",
            "ctr_change_pp",
            "position_change",
            target_col,
        ]
    ].head(10)
)

Modeling pages: 61795
Clients: 34
Features: 12
Missing feature values: 0


,content_id,imp_first_half,avg_daily_imp_early,avg_daily_imp_late,imp_momentum_pct,ctr_change_pp,position_change,is_declining_proxy
0,content_7a105f548d9c6916,4173.0,348.142857,217.000,-37.669265,-0.048929,0.685429,1
1,content_a3ea9792f793ec72,245.0,17.000000,15.750,-7.352941,0.000000,-1.483660,1
2,content_36c36abc7650d7af,3705.0,284.857143,213.875,-24.918506,-0.150451,-0.246919,1
3,content_a7da352b73b02668,2440.0,202.142857,128.125,-36.616608,0.612255,-0.266100,0
4,content_1855a661b4d36130,240.0,15.285714,16.625,8.761682,-0.934579,0.159862,1
5,content_5d412fba6e1a2582,131.0,7.142857,10.125,41.750000,0.000000,-2.508148,1
6,content_1f380a642aed423b,44.0,3.857143,2.125,-44.907407,5.882353,10.932462,0
7,content_22c063002b7c1caf,172.0,12.285714,10.750,-12.500000,0.000000,1.465116,1
8,content_aafb2ab7e5fc80d0,3104.0,193.428571,218.750,13.090842,0.128297,-0.654969,0
9,content_20403327d8d9374c,1294.0,76.285714,95.000,24.531835,0.020205,-0.143327,0


## 3. Split design

I use two protections at the same time:

- **Time separation:** all model features come from March 1–15, while the outcome is measured on March 16–31.
- **Client separation:** around 20% of clients are held out as the final test set, so no held-out client's pages are used for training.

The final test set is frozen before tuning.

Inside the training set only, **GroupKFold** is used for hyperparameter selection. The grouping column is `client_id`, so a client cannot appear in both the training and validation part of a fold.

In [ ]:
TEST_CLIENT_FRACTION = 0.20

# Same style of client holdout as the previous notebook.
clients = model_df["client_id"].drop_duplicates().to_numpy().copy()

rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(clients)

n_test_clients = max(
    1,
    int(round(len(clients) * TEST_CLIENT_FRACTION))
)

test_clients = set(clients[:n_test_clients])
train_clients = set(clients[n_test_clients:])

train_df = model_df[
    model_df["client_id"].isin(train_clients)
].copy()

test_df = model_df[
    model_df["client_id"].isin(test_clients)
].copy()

assert set(train_df["client_id"]).isdisjoint(set(test_df["client_id"]))

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()
groups_train = train_df["client_id"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())
print("Train decline rate:", f"{y_train.mean():.1%}")
print("Test decline rate:", f"{y_test.mean():.1%}")

Train rows: 43817
Test rows: 17978
Train clients: 27
Test clients: 7
Train decline rate: 33.0%
Test decline rate: 31.0%


## 4. Ranking metrics

The main model-selection metric is **Precision@100**.

I use Precision@100 rather than Precision@10 for tuning because 10 pages is a very small sample and can move sharply from only one or two cases. Precision@10 is still reported on the final test set.

In [ ]:
def precision_at_k(y_true, scores, k=100, random_state=RANDOM_STATE):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)

    k = min(k, len(y_true))

    # Break ties randomly instead of relying on row order.
    # Without this, any model that outputs many identical scores
    # (a shallow Decision Tree, or the rule baseline's many 0.0 scores)
    # would have its "top k" decided by the original data order,
    # not by the model itself. That silently biases Precision@k.
    rng = np.random.default_rng(random_state)
    tie_breaker = rng.random(len(scores))
    order = np.lexsort((tie_breaker, -scores))
    top_idx = order[:k]

    return float(y_true[top_idx].mean())


def evaluate_ranking(name, y_true, scores):
    return {
        "model": name,
        "base_rate": float(np.mean(y_true)),
        "precision_at_10": precision_at_k(y_true, scores, 10),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "average_precision": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
    }


def evaluate_threshold(name, y_true, scores, threshold=0.5):
    pred = (np.asarray(scores) >= threshold).astype(int)

    return {
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }

**Fix applied:** `precision_at_k` now breaks tied scores randomly (fixed seed) instead of using row order. Several models here produce many identical scores (e.g. the rule baseline gives a plain 0.0 to every non-matching page, and a shallow Decision Tree repeats the same leaf probability for many rows). Without random tie-breaking, "top k" among tied rows was decided by how the SQL query happened to order the data — not by the model — which could quietly bias every Precision@k number in this notebook.

## 5. Week 4 baseline on the frozen test clients

The rule stays the same:

- good position = average position ≤ 20
- low CTR = CTR below the median for its position bucket
- baseline score = first-half impressions for pages that match both conditions

The position-bucket CTR medians are calculated from **training clients only**, then applied to the test clients.

In [ ]:
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["Top 3", "4-10", "11-20", "21-50", "50+"]

baseline_train = train_df.copy()
baseline_test = test_df.copy()

for frame in (baseline_train, baseline_test):
    frame["position_bucket"] = pd.cut(
        frame["avg_position_first_half"],
        bins=position_bins,
        labels=position_labels,
        include_lowest=True,
    )

train_ctr_medians = (
    baseline_train
    .groupby("position_bucket", observed=True)["ctr_first_half"]
    .median()
)

baseline_test["position_ctr_median"] = (
    baseline_test["position_bucket"]
    .map(train_ctr_medians)
    .astype(float)
    .fillna(baseline_train["ctr_first_half"].median())
)

baseline_test["low_ctr_for_position"] = np.where(
    baseline_test["position_ctr_median"] > 0,
    baseline_test["ctr_first_half"] < baseline_test["position_ctr_median"],
    baseline_test["ctr_first_half"] == 0,
)

baseline_test["good_position"] = (
    baseline_test["avg_position_first_half"] <= 20
)

baseline_test["rule_match"] = (
    baseline_test["good_position"]
    & baseline_test["low_ctr_for_position"]
)

baseline_test["baseline_score"] = np.where(
    baseline_test["rule_match"],
    baseline_test["imp_first_half"],
    0.0,
)

baseline_result = evaluate_ranking(
    "Week 4 Baseline",
    y_test,
    baseline_test["baseline_score"].to_numpy(),
)

display(pd.DataFrame([baseline_result]))

,model,base_rate,precision_at_10,precision_at_100,average_precision,roc_auc
0,Week 4 Baseline,0.309767,0.5,0.47,0.396325,0.618225


## 6. GroupKFold tuning inside training only

This function tries each hyperparameter combination across grouped folds.

No test row is used here.

In [ ]:
N_SPLITS = min(5, train_df["client_id"].nunique())

group_cv = GroupKFold(n_splits=N_SPLITS)


def grouped_parameter_search(
    estimator,
    param_grid,
    X,
    y,
    groups,
    k=100,
):
    rows = []

    for params in ParameterGrid(param_grid):
        fold_scores = []

        for fold, (train_idx, valid_idx) in enumerate(
            group_cv.split(X, y, groups=groups),
            start=1,
        ):
            fold_model = clone(estimator)
            fold_model.set_params(**params)

            X_fold_train = X.iloc[train_idx]
            y_fold_train = y.iloc[train_idx]

            X_fold_valid = X.iloc[valid_idx]
            y_fold_valid = y.iloc[valid_idx]

            fold_model.fit(X_fold_train, y_fold_train)

            valid_scores = fold_model.predict_proba(
                X_fold_valid
            )[:, 1]

            fold_scores.append(
                precision_at_k(
                    y_fold_valid,
                    valid_scores,
                    k=k,
                )
            )

        rows.append({
            "params": dict(params),
            "cv_precision_at_100_mean": np.mean(fold_scores),
            "cv_precision_at_100_std": np.std(fold_scores),
            "fold_scores": fold_scores,
        })

    results = pd.DataFrame(rows)

    results = results.sort_values(
        [
            "cv_precision_at_100_mean",
            "cv_precision_at_100_std",
        ],
        ascending=[False, True],
    ).reset_index(drop=True)

    return results


print("GroupKFold splits:", N_SPLITS)

GroupKFold splits: 5


### Logistic Regression tuning

In [ ]:
logistic_base = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
    ),
])

logistic_grid = {
    "model__C": [0.03, 0.1, 0.3, 1.0, 3.0],
}

logistic_cv = grouped_parameter_search(
    logistic_base,
    logistic_grid,
    X_train,
    y_train,
    groups_train,
)

display(logistic_cv.head(10))

,params,cv_precision_at_100_mean,cv_precision_at_100_std,fold_scores
0,{'model__C': 0.03},0.614,0.143611,"[0.82, 0.39, 0.54, 0.64, 0.68]"
1,{'model__C': 0.1},0.614,0.144305,"[0.83, 0.4, 0.53, 0.63, 0.68]"
2,{'model__C': 0.3},0.614,0.144305,"[0.83, 0.4, 0.53, 0.63, 0.68]"
3,{'model__C': 1.0},0.614,0.144305,"[0.83, 0.4, 0.53, 0.63, 0.68]"
4,{'model__C': 3.0},0.614,0.144305,"[0.83, 0.4, 0.53, 0.63, 0.68]"


### Momentum ablation check

Before looking at the test set, I also compare the Logistic Regression using only level features against the same model family with level + momentum features.

This does not decide the final test result. It checks whether the new direction features add useful validation signal inside the training clients.

In [ ]:
level_only_cols = [
    "log_imp_first_half",
    "log_clicks_first_half",
    "ctr_first_half",
    "avg_position_first_half",
    "active_rate_first_half",
]

logistic_level_cv = grouped_parameter_search(
    logistic_base,
    logistic_grid,
    train_df[level_only_cols].copy(),
    y_train,
    groups_train,
)

momentum_ablation = pd.DataFrame([
    {
        "feature_set": "Level only",
        "best_cv_precision_at_100": logistic_level_cv.iloc[0]["cv_precision_at_100_mean"],
        "cv_std": logistic_level_cv.iloc[0]["cv_precision_at_100_std"],
    },
    {
        "feature_set": "Level + momentum",
        "best_cv_precision_at_100": logistic_cv.iloc[0]["cv_precision_at_100_mean"],
        "cv_std": logistic_cv.iloc[0]["cv_precision_at_100_std"],
    },
])

display(
    momentum_ablation.style.format({
        "best_cv_precision_at_100": "{:.1%}",
        "cv_std": "{:.1%}",
    })
)

,feature_set,best_cv_precision_at_100,cv_std
0,Level only,48.4%,16.1%
1,Level + momentum,61.4%,14.4%


### Decision Tree tuning

In [ ]:
tree_base = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

tree_grid = {
    "max_depth": [3, 5, 8, 12],
    "min_samples_leaf": [50, 150, 300],
}

tree_cv = grouped_parameter_search(
    tree_base,
    tree_grid,
    X_train,
    y_train,
    groups_train,
)

display(tree_cv.head(10))

,params,cv_precision_at_100_mean,cv_precision_at_100_std,fold_scores
0,"{'max_depth': 12, 'min_samples_leaf': 300}",0.622,0.090200,"[0.73, 0.56, 0.49, 0.71, 0.62]"
1,"{'max_depth': 5, 'min_samples_leaf': 300}",0.618,0.123515,"[0.73, 0.6, 0.39, 0.72, 0.65]"
2,"{'max_depth': 5, 'min_samples_leaf': 50}",0.618,0.140342,"[0.76, 0.56, 0.37, 0.7, 0.7]"
3,"{'max_depth': 8, 'min_samples_leaf': 300}",0.606,0.130476,"[0.73, 0.57, 0.37, 0.71, 0.65]"
4,"{'max_depth': 12, 'min_samples_leaf': 50}",0.594,0.144997,"[0.76, 0.54, 0.34, 0.67, 0.66]"
5,"{'max_depth': 8, 'min_samples_leaf': 50}",0.590,0.136675,"[0.76, 0.6, 0.35, 0.67, 0.57]"
6,"{'max_depth': 5, 'min_samples_leaf': 150}",0.584,0.124996,"[0.76, 0.55, 0.38, 0.58, 0.65]"
7,"{'max_depth': 12, 'min_samples_leaf': 150}",0.580,0.120830,"[0.78, 0.52, 0.41, 0.59, 0.6]"
8,"{'max_depth': 8, 'min_samples_leaf': 150}",0.580,0.131149,"[0.78, 0.55, 0.37, 0.59, 0.61]"
9,"{'max_depth': 3, 'min_samples_leaf': 300}",0.476,0.142913,"[0.29, 0.6, 0.39, 0.42, 0.68]"


### Random Forest tuning

In [ ]:
rf_base = RandomForestClassifier(
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_grid = {
    "n_estimators": [300],
    "max_depth": [8, 12],
    "min_samples_leaf": [20, 75],
    "max_features": ["sqrt"],
}

rf_cv = grouped_parameter_search(
    rf_base,
    rf_grid,
    X_train,
    y_train,
    groups_train,
)

display(rf_cv.head(10))

,params,cv_precision_at_100_mean,cv_precision_at_100_std,fold_scores
0,"{'max_depth': 12, 'max_features': 'sqrt', 'min...",0.670,0.133417,"[0.87, 0.56, 0.49, 0.73, 0.7]"
1,"{'max_depth': 8, 'max_features': 'sqrt', 'min_...",0.670,0.132816,"[0.87, 0.58, 0.48, 0.73, 0.69]"
2,"{'max_depth': 8, 'max_features': 'sqrt', 'min_...",0.664,0.130323,"[0.86, 0.59, 0.47, 0.72, 0.68]"
3,"{'max_depth': 12, 'max_features': 'sqrt', 'min...",0.650,0.141563,"[0.85, 0.55, 0.44, 0.71, 0.7]"


## 7. Fit the best CV version of each model on all training clients

Only now do we fit the selected hyperparameters on the full training set.

In [ ]:
def best_params_from_search(search_df):
    return dict(search_df.iloc[0]["params"])


logistic_best_params = best_params_from_search(logistic_cv)
tree_best_params = best_params_from_search(tree_cv)
rf_best_params = best_params_from_search(rf_cv)

cv_selection_table = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "cv_precision_at_100_mean": logistic_cv.iloc[0]["cv_precision_at_100_mean"],
        "cv_precision_at_100_std": logistic_cv.iloc[0]["cv_precision_at_100_std"],
    },
    {
        "model": "Decision Tree",
        "cv_precision_at_100_mean": tree_cv.iloc[0]["cv_precision_at_100_mean"],
        "cv_precision_at_100_std": tree_cv.iloc[0]["cv_precision_at_100_std"],
    },
    {
        "model": "Random Forest",
        "cv_precision_at_100_mean": rf_cv.iloc[0]["cv_precision_at_100_mean"],
        "cv_precision_at_100_std": rf_cv.iloc[0]["cv_precision_at_100_std"],
    },
]).sort_values(
    "cv_precision_at_100_mean",
    ascending=False,
).reset_index(drop=True)

selected_model_name = cv_selection_table.iloc[0]["model"]

print("Best Logistic params:", logistic_best_params)
print("Best Tree params:", tree_best_params)
print("Best RF params:", rf_best_params)
print()
print("Model selected BEFORE opening the final test set:", selected_model_name)

display(
    cv_selection_table.style.format({
        "cv_precision_at_100_mean": "{:.1%}",
        "cv_precision_at_100_std": "{:.1%}",
    })
)

best_logistic = clone(logistic_base).set_params(**logistic_best_params)
best_tree = clone(tree_base).set_params(**tree_best_params)
best_rf = clone(rf_base).set_params(**rf_best_params)

best_logistic.fit(X_train, y_train)
best_tree.fit(X_train, y_train)
best_rf.fit(X_train, y_train)

print("Final training complete.")

Best Logistic params: {'model__C': 0.03}
Best Tree params: {'max_depth': 12, 'min_samples_leaf': 300}
Best RF params: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 75, 'n_estimators': 300}

Model selected BEFORE opening the final test set: Random Forest


,model,cv_precision_at_100_mean,cv_precision_at_100_std
0,Random Forest,67.0%,13.3%
1,Decision Tree,62.2%,9.0%
2,Logistic Regression,61.4%,14.4%


Final training complete.


## 8. Final test evaluation — one time

The test set is used here after feature design and hyperparameter selection are finished.

**Caveat:** the frozen test set has only 7 clients. Precision@100 on this test set is a single point estimate, not a stable average — a couple of unusual clients can move it a lot. Treat the model-vs-baseline comparison as directional, not exact, until it is checked against more clients or more time periods.

In [ ]:
model_objects = {
    "Logistic Regression": best_logistic,
    "Decision Tree": best_tree,
    "Random Forest": best_rf,
}

ranking_results = [baseline_result]
threshold_results = []
test_scores = {}

for name, model in model_objects.items():
    scores = model.predict_proba(X_test)[:, 1]
    test_scores[name] = scores

    ranking_results.append(
        evaluate_ranking(name, y_test, scores)
    )

    threshold_results.append(
        evaluate_threshold(name, y_test, scores)
    )

comparison_table = (
    pd.DataFrame(ranking_results)
    .sort_values("precision_at_100", ascending=False)
    .reset_index(drop=True)
)

threshold_table = pd.DataFrame(threshold_results)

display(
    comparison_table.style.format({
        "base_rate": "{:.1%}",
        "precision_at_10": "{:.1%}",
        "precision_at_100": "{:.1%}",
        "average_precision": "{:.3f}",
        "roc_auc": "{:.3f}",
    })
)

display(
    threshold_table.style.format({
        "accuracy": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}",
    })
)

,model,base_rate,precision_at_10,precision_at_100,average_precision,roc_auc
0,Random Forest,31.0%,80.0%,79.0%,0.569,0.737
1,Logistic Regression,31.0%,90.0%,78.0%,0.550,0.724
2,Decision Tree,31.0%,70.0%,72.0%,0.542,0.718
3,Week 4 Baseline,31.0%,50.0%,47.0%,0.396,0.618


,model,accuracy,precision,recall,f1
0,Logistic Regression,0.693,0.504,0.636,0.562
1,Decision Tree,0.660,0.463,0.615,0.528
2,Random Forest,0.678,0.484,0.596,0.534


In [ ]:
selected_test_p100 = (
    comparison_table
    .set_index("model")
    .loc[selected_model_name, "precision_at_100"]
)

baseline_test_p100 = (
    comparison_table
    .set_index("model")
    .loc["Week 4 Baseline", "precision_at_100"]
)

print(
    f"Pre-selected model from GroupKFold CV: {selected_model_name}"
)

print(
    f"Final held-out Precision@100: {selected_test_p100:.1%}"
)

print(
    f"Week 4 baseline Precision@100: {baseline_test_p100:.1%}"
)

print(
    f"Absolute lift vs baseline: "
    f"{(selected_test_p100 - baseline_test_p100):+.1%}"
)

Pre-selected model from GroupKFold CV: Random Forest
Final held-out Precision@100: 79.0%
Week 4 baseline Precision@100: 47.0%
Absolute lift vs baseline: +32.0%


## 9. Error analysis and interpretation

The ranking metric tells me whether the queue is useful. It does not tell me why the model is wrong.

I will inspect:

- the top-ranked false positives: pages the model considered high-risk that did not cross the decline threshold;
- the lowest-ranked false negatives: pages that declined but received relatively low risk scores;
- permutation importance to see which features the selected model depends on most.

These checks happen after model selection. They are interpretation, not tuning.

In [ ]:
best_model = model_objects[selected_model_name]
best_scores = test_scores[selected_model_name]

error_df = test_df[
    [
        "content_id",
        target_col,
        "avg_daily_imp_early",
        "avg_daily_imp_late",
        "imp_momentum_pct",
        "ctr_first_half",
        "ctr_change_pp",
        "avg_position_first_half",
        "position_change",
    ]
].copy()

error_df["risk_score"] = best_scores

top_false_positives = (
    error_df[error_df[target_col] == 0]
    .sort_values("risk_score", ascending=False)
    .head(3)
)

missed_declines = (
    error_df[error_df[target_col] == 1]
    .sort_values("risk_score", ascending=True)
    .head(3)
)

print("Three high-confidence false positives:")
display(top_false_positives)

print("Three low-ranked true declines:")
display(missed_declines)

Three high-confidence false positives:


,content_id,is_declining_proxy,avg_daily_imp_early,avg_daily_imp_late,imp_momentum_pct,ctr_first_half,ctr_change_pp,avg_position_first_half,position_change,risk_score
10828,content_27edb2e75a92802a,0,116.714286,22.625,-80.615055,0.0,0.0,6.625251,19.334312,0.903990
9893,content_588f7d5831396d54,0,61.142857,11.375,-81.396028,0.0,0.0,5.077071,13.338605,0.902128
9618,content_ac671311da8857d9,0,292.285714,19.250,-93.413978,0.0,0.0,3.547727,9.751780,0.901431


Three low-ranked true declines:


,content_id,is_declining_proxy,avg_daily_imp_early,avg_daily_imp_late,imp_momentum_pct,ctr_first_half,ctr_change_pp,avg_position_first_half,position_change,risk_score
45604,content_cd02807820914877,1,3.714286,5.125,37.980769,0.0,0.0,5.089552,-2.870544,0.133537
51888,content_60a962a7d638259a,1,3.142857,4.125,31.250000,0.0,0.0,15.854545,1.348485,0.135860
10017,content_ea2c010b2afdaa96,1,3.571429,5.875,64.500000,0.0,0.0,52.305556,13.336170,0.137269


In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(12))

,feature,importance_mean,importance_std
0,imp_momentum_log,0.069412,0.002983
1,log_imp_first_half,0.065406,0.003712
2,imp_momentum_pct,0.032317,0.002558
3,ctr_first_half,0.027016,0.001482
4,avg_position_first_half,0.019013,0.001001
5,log_clicks_first_half,0.010440,0.001661
6,click_change_per_day,0.001863,0.000579
7,ctr_change_pp,0.001235,0.000386
8,active_rate_first_half,0.000000,0.000000
9,active_rate_change,0.000000,0.000000


In [ ]:
# Compact summary for the notebook conclusion.
top_features = importance_df.head(3)["feature"].tolist()

print("Top 3 features:", ", ".join(top_features))
print()
print(
    "Interpretation check: make sure the strongest features are plausible "
    "pre-outcome signals and not fields derived from March 16-31."
)

Top 3 features: imp_momentum_log, log_imp_first_half, imp_momentum_pct

Interpretation check: make sure the strongest features are plausible pre-outcome signals and not fields derived from March 16-31.


##Retraining With Random split :

In [ ]:
print(rf_best_params)

{'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 75, 'n_estimators': 300}


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

# Same training data used by the grouped CV
X_random_cv = X_train.copy()
y_random_cv = y_train.copy()

random_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

random_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(
    random_cv.split(X_random_cv, y_random_cv), start=1
):
    X_tr = X_random_cv.iloc[train_idx]
    X_val = X_random_cv.iloc[val_idx]
    y_tr = y_random_cv.iloc[train_idx]
    y_val = y_random_cv.iloc[val_idx]

    model = RandomForestClassifier(
        max_depth=12,
        max_features="sqrt",
        min_samples_leaf=75,
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(X_tr, y_tr)

    scores = model.predict_proba(X_val)[:, 1]

    # Precision@100
    k = min(100, len(y_val))
    top_idx = np.argsort(scores)[::-1][:k]
    precision_at_100 = y_val.iloc[top_idx].mean()

    random_fold_scores.append(precision_at_100)

    print(f"Fold {fold}: Precision@100 = {precision_at_100:.1%}")

print()
print(
    "Random 5-fold CV Precision@100:",
    f"{np.mean(random_fold_scores):.1%} ± {np.std(random_fold_scores):.1%}"
)

print(
    "Grouped 5-fold CV Precision@100:",
    "67.0% ± 13.3%"
)

Fold 1: Precision@100 = 81.0%
Fold 2: Precision@100 = 80.0%
Fold 3: Precision@100 = 77.0%
Fold 4: Precision@100 = 83.0%
Fold 5: Precision@100 = 76.0%

Random 5-fold CV Precision@100: 79.4% ± 2.6%
Grouped 5-fold CV Precision@100: 67.0% ± 13.3%


##Leakage audit

In [ ]:
# Leakage audit for the final feature set

forbidden_exact = {
    target_col,
    "impression_change_pct",
    "imp_future",
    "clicks_future",
    "sum_position_future",
    "active_days_future",
    "avg_daily_imp_future",
    "client_id",
    "content_id",
}

# Anything explicitly referring to the future/outcome window
future_features = [
    col for col in feature_cols
    if "future" in col.lower()
]

# Exact forbidden columns accidentally used as features
forbidden_in_features = [
    col for col in feature_cols
    if col in forbidden_exact
]

print("Number of model features:", len(feature_cols))
print("Future-window features found:", future_features)
print("Forbidden/target-derived features found:", forbidden_in_features)

if not future_features and not forbidden_in_features:
    print("\nPASS: No direct future-window, target-derived, ID, or outcome columns found.")
else:
    print("\nWARNING: Potential leakage found. Review the columns above.")

print("\nFinal feature set:")
for col in feature_cols:
    print("-", col)

Number of model features: 12
Future-window features found: []
Forbidden/target-derived features found: []

PASS: No direct future-window, target-derived, ID, or outcome columns found.

Final feature set:
- log_imp_first_half
- log_clicks_first_half
- ctr_first_half
- avg_position_first_half
- active_rate_first_half
- imp_momentum_pct
- imp_momentum_log
- click_change_per_day
- ctr_change_pp
- position_change
- active_rate_change
- has_position_momentum


## 10. What I will conclude after running the notebook

I will report the result only after the final test cell runs.

The claim should stay narrow:

- the model predicts the defined **decline proxy**, not Google's algorithm;
- the result is measured on held-out clients;
- hyperparameters were selected with GroupKFold inside training only;
- momentum features use only information available before the future outcome window;
- if the model does not beat the baseline, that is still a valid result and will be reported.

### Self-check

- [ ] Notebook runs top to bottom
- [ ] No future-window column appears in `feature_cols`
- [ ] No client ID or content ID is used as a model feature
- [ ] Test clients are disjoint from training clients
- [ ] GroupKFold uses `client_id` as the group
- [ ] Hyperparameters are chosen from CV, not from the test score
- [ ] Baseline and models use the same final test clients
- [ ] Precision@100 is reported for baseline and all models
- [ ] Error cases and top features are inspected
- [ ] Final claims use observed / measured / directional / decision-support language